# Compositional Analysis

**REQUIRED DAY 3**

## A different question from lesson 04

Lesson 04 asked: within one cell type, did gene expression change? This lesson asks something structurally different: did the *mix* of cell types itself shift with stimulation? Same dataset, same donor-paired design, a genuinely different statistical question — and one with a trap built into the data itself.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

adata = sc.read_h5ad("/tscc/nfs/home/juf009/day3_shared_data/kang_2018_checkpoint.h5ad")

counts = adata.obs.groupby(["replicate", "label", "cell_type"], observed=True).size().reset_index(name="n")
totals = counts.groupby(["replicate", "label"], observed=True)["n"].transform("sum")
counts["prop"] = counts["n"] / totals

stim = counts[counts["label"] == "stim"].pivot(index="replicate", columns="cell_type", values="prop").fillna(0)
ctrl = counts[counts["label"] == "ctrl"].pivot(index="replicate", columns="cell_type", values="prop").fillna(0)
stim, ctrl = stim.align(ctrl, join="inner")
stim.shape  # (8 donors, 8 cell types)

## The trap: proportions sum to 1

Every donor's cell-type proportions add up to exactly 1, in both conditions — check it yourself:

In [ ]:
pd.concat([stim.sum(axis=1).rename("stim_total"), ctrl.sum(axis=1).rename("ctrl_total")], axis=1)

That constraint isn't just bookkeeping — it means cell types **cannot** vary independently. If one type's share goes up, something else's share must go down, mechanically, whether or not there's any real biology behind it. Testing each cell type's proportion as if it were an independent measurement (a plain t-test per type, ignoring the others) treats this mechanical coupling as if it were real, independent signal. Check the actual correlation structure in this data:

In [ ]:
delta = stim - ctrl
delta.corr().round(2)

Real numbers here: CD4 T cells and CD14+ Monocytes shift with a **-0.65 correlation** across donors — when one goes up, the other tends to go down, across this whole dataset, purely from the sum-to-one constraint (plus whatever real biology is layered on top). Agent-B item 21 exists precisely for this: does the analysis account for this coupling, or pretend each proportion is free-floating?

## The actual test: paired, per cell type, corrected across all of them

Given the paired donor structure (Day 3's recurring theme), a **paired Wilcoxon signed-rank test** per cell type, across the 8 donors, is a reasonable dependency-free approach — followed by multiple-testing correction across all cell types tested, not just the one you're interested in.

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

results = []
for ct in stim.columns:
    stat, p = wilcoxon(stim[ct], ctrl[ct])
    results.append({"cell_type": ct, "median_stim": stim[ct].median(), "median_ctrl": ctrl[ct].median(), "pval": p})
res_df = pd.DataFrame(results)
res_df["padj"] = multipletests(res_df["pval"], method="fdr_bh")[1]
res_df.sort_values("padj")

## The real, honest result

**Nothing survives correction.** CD14+ Monocytes comes closest — raw p=0.016, but `padj=0.125` after correcting across all 8 cell types — which does not clear the conventional 0.05 threshold.

This is not a failed analysis. It's a real, informative result: **the transcriptional response in lesson 04 was dramatic (thousands of genes), but the cell-type *composition* did not detectably shift in this 6-hour window.** Those are different levels of biological response, and a 6-hour interferon exposure is plausibly enough time to reprogram gene expression inside existing cells without producing a measurable change in how many of each cell type are present. Resist the pull to keep p-hacking cell-type definitions or dropping the correction until something clears 0.05 — the honest answer here is "no detectable compositional shift," and that is itself worth reporting.

## Further reading

For a real study, `scCODA` (Bayesian, matches the field's reference textbook) or the newer, lighter `scanpro` are the tools to reach for once you need more power or a more principled model than a paired Wilcoxon test — neither is installed in `mstp-day3` (both add real dependency weight; `scCODA` in particular needs `tensorflow`), so they're mentioned here as where to go next, not run today.

- [Single-cell best practices — Compositional analysis](https://www.sc-best-practices.org/conditions/compositional.html)
- [scCODA documentation](https://sccoda.readthedocs.io/)
- [scanpro (Scientific Reports, 2024)](https://www.nature.com/articles/s41598-024-66381-7)

## Practice

Run the correlation matrix and the corrected test table yourself. Open a fresh Agent B session and run checklist item 21 (sum-to-one accounted for) and item 22 (pairing preserved) against this notebook.